### RAG pipeline  
Load, clean, chunk, index.

Source text: PDFs from UK Financial Conduct Authority (FCA) online [Handbook](https://handbook.fca.org.uk/handbook), specifically the Conduct of Business Sourcebook (COBS) section, chapters 1-10A, last updated on 5 August 2026. 

In [0]:
%pip install \
    langchain==1.3.16 \
    langchain-chroma==1.1.0 \
    langchain-groq==1.1.3 \
    langchain-huggingface==1.2.2 \
    huggingface_hub==1.28.0 \
    sentence-transformers==6.0.0 \
    langchain_community \
    pypdf \
    torch==2.13.0 \
    torchvision==0.28.0 \
    torchaudio==2.11.0 \
    faiss-cpu==1.15.0
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader # langchain-community sunsetting, fix this later
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

DATA_PATH = "fca_cobs_pdfs"
SAVE_PATH = "faiss_index" 
MODEL_NAME = "all-MiniLM-L6-v2"

# load PDFs
loader = PyPDFDirectoryLoader(DATA_PATH)
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDFs.")

In [0]:
def clean_fca_text(text):
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        # skip lines that are just R, G, N or COBS (case insensitive)
        if line.upper() in ['R', 'G', 'N', 'COBS', 'CHAPTER']:
            continue
        # skip lines that are just dots or whitespace
        if not line or re.match(r'^[\.\s\-_]+$', line):
            continue
        # skip lines that look like the handbook header/footer
        if 'www.handbook.fca.org.uk' in line or re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s\d{4}', line):
            continue
        cleaned_lines.append(line)
    # join back with single newlines
    return '\n'.join(cleaned_lines)

In [0]:
for doc in docs:
    doc.page_content = clean_fca_text(doc.page_content)

# chunk documents
# 1000 character chunk with 200 overlap to keep legal context intact
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")

# spotcheck
import random
print(random.choice(chunks).page_content)

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# initialise embedding model
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)

# FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("faiss_index")
print(f"Vector store successfully saved.")
